In [0]:
# ================= 1. DIMENSIONS & TYPES =================
dbutils.notebook.run("./01_rule_type", 60, {"target_environment": "dev", "rule_type": "technical", "description": "DQ", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./01_rule_type", 60, {"target_environment": "dev", "rule_type": "business", "description": "Business Rules", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./02_rule_dimension", 60, {"target_environment": "dev", "rule_dimension": "validity", "description": "Business logic", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./02_rule_dimension", 60, {"target_environment": "dev", "rule_dimension": "uniqueness", "description": "Duplicates", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./03_project", 60, {"target_environment": "dev", "project": "demo_project", "project_description": "Demo", "added_by": "Demo", "action": "insert/update"})

# ================= 2. POLICIES =================
# Policy A: Error & Quarantine (For single failures)
dbutils.notebook.run("./04_run_policy", 60, {"target_environment": "dev", "criticality": "high", "category": "error", "quarantine": "true", "added_by": "Demo", "action": "insert/update"})
# Policy B: Warning No Quarantine (For whole column nulls)
dbutils.notebook.run("./04_run_policy", 60, {"target_environment": "dev", "criticality": "low", "category": "warning", "quarantine": "false", "added_by": "Demo", "action": "insert/update"})

# ================= 3. APPLY_AT =================
dbutils.notebook.run("./05_apply_at", 60, {"target_environment": "dev", "apply_at": "gold", "is_project_specific": "false", "added_by": "Demo", "action": "insert/update"})

# ================= 4. RULE TEMPLATES =================
# Template 1: DQX Is Not Null
dbutils.notebook.run("./06_rule_template", 60, {"target_environment": "dev", "name": "is_not_null", "description": "value is not null.", "rule_type": "technical", "rule_dimension": "validity", "scope": "column", "is_reusable": "true", "engine_type": "dqx", "statement": "is_not_null", "added_by": "Demo", "action": "insert/update"})

# Template 2: Primary Key Violation
dbutils.notebook.run("./06_rule_template", 60, {"target_environment": "dev", "name": "primary_key_violation", "description": "Primary Key Violation", "rule_type": "technical", "rule_dimension": "uniqueness", "scope": "table", "is_reusable": "true", "engine_type": "sql", "statement": "select ${i:columns} from ${table} where 1=1 group by ${i:columns} having count(*) > 1", "added_by": "Demo", "action": "insert/update"})

# Template 3: Threshold Range Check (Business Rule)
dbutils.notebook.run("./06_rule_template", 60, {"target_environment": "dev", "name": "check_thresholds_dynamic", "description": "Value must be between min and max", "rule_type": "business", "rule_dimension": "validity", "scope": "column", "is_reusable": "true", "engine_type": "sql", "statement": "SELECT * FROM ${table} WHERE ${i:column} < ${v:min_val} OR ${i:column} > ${v:max_val}", "added_by": "Demo", "action": "insert/update"})

# ================= 5. TABLE REGISTRATION =================
# Register ALL THREE tables!
dbutils.notebook.run("./07_table", 60, {"target_environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips", "layer": "gold", "primary_keys": "trip_id", "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./07_table", 60, {"target_environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips_clean", "layer": "gold", "primary_keys": "trip_id", "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./07_table", 60, {"target_environment": "dev", "catalog": "analytics_dq_dev", "schema": "metadata", "table": "test_trips_warning", "layer": "gold", "primary_keys": "trip_id", "filter_field": "tpep_dropoff_datetime", "filter_field_type": "timestamp", "added_by": "Demo", "action": "insert/update"})

# ================= 6. RULE ASSIGNMENTS =================

# -----> ASSIGNMENTS FOR THE DIRTY TABLE (test_trips)
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips", "template": "is_not_null", "policy": "error.high.quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "fare_amount"}', "parameters_values": "", "project": "demo_project", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips", "template": "is_not_null", "policy": "warning.low.no_quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "dropoff_zip"}', "parameters_values": "", "project": "demo_project", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips", "template": "primary_key_violation", "policy": "error.high.quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"columns": ["trip_id"]}', "parameters_values": "", "project": "demo_project", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips", "template": "check_thresholds_dynamic", "policy": "error.high.quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "trip_distance"}', "parameters_values": '{"min_val": "0", "max_val": "100"}', "project": "demo_project", "added_by": "Demo", "action": "insert/update"})

# -----> ASSIGNMENTS FOR THE CLEAN TABLE (test_trips_clean)
# We apply two rules that will pass 100%
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips_clean", "template": "is_not_null", "policy": "error.high.quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "fare_amount"}', "parameters_values": "", "project": "demo_project", "added_by": "Demo", "action": "insert/update"})
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips_clean", "template": "check_thresholds_dynamic", "policy": "error.high.quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "trip_distance"}', "parameters_values": '{"min_val": "0", "max_val": "100"}', "project": "demo_project", "added_by": "Demo", "action": "insert/update"})

# -----> ASSIGNMENTS FOR THE WARNING TABLE (test_trips_warning)
# We only apply the rule that produces Warnings
dbutils.notebook.run("./08_rule_assignment", 60, {"target_environment": "dev", "table": "analytics_dq_dev.metadata.test_trips_warning", "template": "is_not_null", "policy": "warning.low.no_quarantine", "apply_at": "gold.project_agnostic", "parameters_identifiers": '{"column": "dropoff_zip"}', "parameters_values": "", "project": "demo_project", "added_by": "Demo", "action": "insert/update"})


print("✅ Demo setup completed for ALL THREE tables!")